# Get a Certificate for accessing the CEDA Archive

We want to access CRU TS v. 4.04 gridded datasets from the [CEDA](https://catalogue.ceda.ac.uk/uuid/89e1e34ec3554dc98594a5732622bce9) database.

In [ ]:
!pip install ContrailOnlineCAClient netCDF4

In [ ]:
import intake

catalog_url = "master.yaml"
catalog = intake.open_catalog(catalog_url)
atmo_cat = catalog.atmosphere
list(atmo_cat)

We need to get a certificate for data access to `CRU_TS`, we take the example code from [here](https://github.com/cedadev/opendap-python-example/blob/master/remote_nc_reader.py), to create a certificate. Once, the certificate is created on your machine, you can access the data. The certification has to be done only once per machine.

In [ ]:
import datetime
import os
from getpass import getpass

from contrail.security.onlineca.client import OnlineCaClient

# Import third-party libraries
from cryptography import x509
from cryptography.hazmat.backends import default_backend

# Credentials defaults
DODS_FILE_CONTENTS = """HTTP.COOKIEJAR=./dods_cookies
HTTP.SSL.CERTIFICATE=./credentials.pem
HTTP.SSL.KEY=./credentials.pem
HTTP.SSL.CAPATH=./ca-trustroots
"""

DODS_FILE_PATH = os.path.expanduser("~/.dodsrc")
CERTS_DIR = os.path.expanduser("~/.certs")

if not os.path.isdir(CERTS_DIR):
    os.makedirs(CERTS_DIR)

TRUSTROOTS_DIR = os.path.join(CERTS_DIR, "ca-trustroots")
CREDENTIALS_FILE_PATH = os.path.join(CERTS_DIR, "credentials.pem")

TRUSTROOTS_SERVICE = "https://slcs.ceda.ac.uk/onlineca/trustroots/"
CERT_SERVICE = "https://slcs.ceda.ac.uk/onlineca/certificate/"


def write_dods_file_contents():
    DODS_FILE_CONTENTS = """
    HTTP.COOKIEJAR=./dods_cookies
    HTTP.SSL.CERTIFICATE={credentials_file_path}
    HTTP.SSL.KEY={credentials_file_path}
    HTTP.SSL.CAPATH={trustroots_dir}
    """.format(
        credentials_file_path=CREDENTIALS_FILE_PATH, trustroots_dir=TRUSTROOTS_DIR
    )

    with open(DODS_FILE_PATH, "w") as dods_file:
        dods_file.write(DODS_FILE_CONTENTS)


def cert_is_valid(cert_file, min_lifetime=0):
    """
    Returns boolean - True if the certificate is in date.
    Optional argument min_lifetime is the number of seconds
    which must remain.
    :param cert_file: certificate file path.
    :param min_lifetime: minimum lifetime (seconds)
    :return: boolean
    """
    try:
        with open(cert_file, "rb") as f:
            crt_data = f.read()
    except IOError:
        return False

    try:
        cert = x509.load_pem_x509_certificate(crt_data, default_backend())
    except ValueError:
        return False

    now = datetime.datetime.now()

    return (
        cert.not_valid_before <= now
        and cert.not_valid_after > now + datetime.timedelta(0, min_lifetime)
    )


def setup_credentials(force=False):
    """
    Download and create required credentials files.
    Return True if credentials were set up.
    Return False is credentials were already set up.
    :param force: boolean
    :return: boolean
    """
    # Test for DODS_FILE and only re-get credentials if it doesn't
    # exist AND `force` is True AND certificate is in-date.
    if (
        os.path.isfile(DODS_FILE_PATH)
        and not force
        and cert_is_valid(CREDENTIALS_FILE_PATH)
    ):
        print("[INFO] Security credentials already set up.")
        return False

    onlineca_client = OnlineCaClient()
    onlineca_client.ca_cert_dir = TRUSTROOTS_DIR

    # Set up trust roots
    onlineca_client.get_trustroots(
        TRUSTROOTS_SERVICE, bootstrap=True, write_to_ca_cert_dir=True
    )

    username = input("CEDA username")  # os.environ['CEDA_USERNAME']
    password = getpass("CEDA password")  # os.environ['CEDA_PASSWORD']

    # Write certificate credentials file
    key_pair, certs = onlineca_client.get_certificate(
        username, password, CERT_SERVICE, pem_out_filepath=CREDENTIALS_FILE_PATH
    )

    # Write the dodsrc credentials file
    write_dods_file_contents()

    print("[INFO] Security credentials set up.")
    return True

Now, we can simply run the `setup_credentials`

In [ ]:
setup_credentials(force=True)

Great! Now we should be able to discover the `CRU_TS` dataset!

In [ ]:
list(atmo_cat.CRU_TS)

In [ ]:
atmo_cat.CRU_TS.CRU_TS4_04.discover()

Let's access the temperature variable of the latest version to check if everything works:

In [ ]:
cru_tmp_ds = atmo_cat.CRU_TS.CRU_TS4_04.get(variable="tmp").read_chunked()
cru_tmp_ds

In [ ]:
cru_tmp_ds.tmp